# XGBoost

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd

import xgboost as xgb

In [2]:
SEED = 10

In [3]:
np.random.seed(SEED)
_ = torch.manual_seed(SEED)

## Dataset

In [4]:
from dataset_11t5g_markov import DatasetMarkovSurvSurf

ds_name = 'markov_32feat_11t5g_less_balanced'
ds_df_dir = '/home/yc366/repos/survsurf_benchmark/dataset_split'
project = 'SurvSurfBenchmark_Markov_imbalanced'
SAVE_PATH = f'./xgboost_models/{project}/xgboost_mono_{SEED}'
g_resol = 1
t_resol = 1
split = 'train'
ds = DatasetMarkovSurvSurf(
    df_dir=ds_df_dir, 
    ds_name=ds_name, 
    g_resol=g_resol, 
    split=split, 
    mode='first_cross_obs_only', 
    separate_g_from_feats=False
)
df_train = ds._get_df_Xy_trans_obs()

split = 'train'
ds = DatasetMarkovSurvSurf(
    df_dir=ds_df_dir, 
    ds_name=ds_name, 
    g_resol=g_resol, 
    split=split, 
    mode='true_probs_grid', 
    separate_g_from_feats=False
)
df_train_pred = ds._get_df_Xy_true_prob()

split = 'val'
ds = DatasetMarkovSurvSurf(
    df_dir=ds_df_dir, 
    ds_name=ds_name, 
    g_resol=g_resol, 
    split=split, 
    mode='true_probs_grid', 
    separate_g_from_feats=False
)
df_val = ds._get_df_Xy_true_prob()

In [5]:
df_train.head()

,subject,event_observed,duration,g_max_by_time,feat00,feat01,feat02,feat03,feat04,feat05,...,feat24,feat25,feat26,feat27,feat28,feat29,feat30,feat31,weight,is_t_trans
0,0,0,8,1.0,0.883893,0.195865,0.357537,-2.343262,-1.084833,0.559696,...,-0.243339,-0.130031,-0.109017,1.556186,0.128778,-2.066949,-0.885493,0.0,1,1
1,2,1,2,1.0,1.301429,-0.994802,2.094665,0.359429,0.691002,-1.485720,...,-0.938910,-1.901464,1.095433,-0.333544,-1.441849,-1.416667,-1.629502,0.0,1,1
2,2,0,9,2.0,1.301429,-0.994802,2.094665,0.359429,0.691002,-1.485720,...,-0.938910,-1.901464,1.095433,-0.333544,-1.441849,-1.416667,-1.629502,0.0,1,1
3,4,0,10,1.0,-0.105813,2.009902,0.011231,-1.089429,-1.514511,-0.559761,...,-1.941266,0.341435,1.862855,-0.295230,0.478317,-0.715190,1.237293,1.0,1,1
4,6,1,7,1.0,0.805669,-0.012551,-0.260823,-0.889478,-1.615899,-0.851965,...,-0.708124,1.414139,-0.490533,1.181171,-0.046925,-1.460106,-0.508191,0.0,1,1


## Feature transforms

In [6]:
x_train = df_train.loc[:,df_train.columns.str.startswith('feat')|df_train.columns.str.startswith('g_max_by_time')].astype('float32')
x_train['g_max_by_time'] = x_train['g_max_by_time']/ds.g_max
mono_constraints = {i:0 if i != 'g_max_by_time' else -1 for i in x_train.columns}
x_train = x_train


x_train_pred = df_train_pred.loc[:,df_train_pred.columns.str.startswith('feat')|df_train_pred.columns.str.startswith('g_max_by_time')].astype('float32')
x_train_pred['g_max_by_time'] = x_train_pred['g_max_by_time']/ds.g_max
x_train_pred = x_train_pred


x_val = df_val.loc[:,df_val.columns.str.startswith('feat')|df_val.columns.str.startswith('g_max_by_time')].astype('float32')
x_val['g_max_by_time'] = x_val['g_max_by_time']/ds.g_max
x_val = x_val

assert all(df_train.loc[:,df_train.columns.str.startswith('feat')|df_train.columns.str.startswith('g_max_by_time')].columns == (
    df_val.loc[:,df_val.columns.str.startswith('feat')|df_val.columns.str.startswith('g_max_by_time')].columns
))



In [7]:
import xgboost as xgb
y_train = df_train['duration']*np.where(df_train['event_observed']==1, 1, -1)


params = {
    'objective': 'survival:cox',
    'monotone_constraints': mono_constraints,
    'random_state':10
}
dtrain = xgb.DMatrix(x_train, label=y_train)
xgboost_model = xgb.train(params, dtrain)
from lifelines import CoxPHFitter

train_results = pd.DataFrame({
    'xgb_risk': np.log(xgboost_model.predict(dtrain)),
    'time': df_train['duration'],
    'event': df_train['event_observed']
})

# Fit a simple Cox model on the XGBoost scores to get the baseline hazard (a quick work-around re-implementing Breslow)
cph = CoxPHFitter()
cph.fit(train_results, duration_col='time', event_col='event')


<lifelines.CoxPHFitter: fitted with 1481 total observations, 1000 right-censored observations>

## Prediction

In [8]:
df_val['event_observed'].value_counts()

event_observed
0.000000e+00    7500
1.000000e+00      75
1.000000e+00      44
1.000000e+00      31
1.000000e+00      24
                ... 
2.180312e-10       1
2.654184e-02       1
5.588085e-06       1
2.138013e-09       1
4.691168e-02       1
Name: count, Length: 19809, dtype: int64

In [9]:
df_train_pred['event_observed'].value_counts()

event_observed
0.000000e+00    15000
1.000000e+00       89
1.000000e+00       59
1.000000e+00       55
1.000000e+00       47
                ...  
7.690267e-05        1
2.164046e-01        1
9.938195e-01        1
9.940693e-09        1
1.216277e-06        1
Name: count, Length: 39709, dtype: int64

In [10]:
dval = xgb.DMatrix(x_val)
# dtrain = xgb.DMatrix(x_train, label=y_train)
from lifelines import CoxPHFitter

val_results = pd.DataFrame({
    'xgb_risk': np.log(xgboost_model.predict(dval)),
})
pred_grid_val = 1-cph.predict_survival_function(val_results).T



In [11]:
from sklearn.metrics import mean_squared_error
df_val_pred = df_val.copy()
df_val_pred['pred'] = [np.interp(x=t, xp=pred_grid_val.columns, fp=pred_grid_val.loc[idx,:]) for idx, t in df_val_pred['duration'].items()]
mean_squared_error(y_true=df_val_pred['event_observed'], y_pred=df_val_pred['pred'])

0.038498728058054744

In [12]:
dtrain_pred = xgb.DMatrix(x_train_pred)
# dtrain = xgb.DMatrix(x_train, label=y_train)
from lifelines import CoxPHFitter

train_pred_results = pd.DataFrame({
    'xgb_risk': np.log(xgboost_model.predict(dtrain_pred)),
})
pred_grid_train = 1-cph.predict_survival_function(train_pred_results).T

In [13]:
df_train_pred_all = df_train_pred.copy()
df_train_pred_all['pred'] = [np.interp(x=t, xp=pred_grid_train.columns, fp=pred_grid_train.loc[idx,:]) for idx, t in df_train_pred_all['duration'].items()]
mean_squared_error(y_true=df_train_pred_all['event_observed'], y_pred=df_train_pred_all['pred'])

0.01762577003647622

In [14]:
import pickle
xgboost_model.save_model(f'{SAVE_PATH}_trees.ubj')


with open(f'{SAVE_PATH}_cph.pkl', 'wb') as fp:
    pickle.dump(cph, file=fp)